# NLP Movie Classification (movies_final_cleaned.csv)

Полный pipeline: загрузка данных, предобработка текста, тематическое моделирование, кластеризация и классификация.

## 1. Data Loading & Exploration

In [ ]:
import re
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords

import pymorphy3
from sklearn.decomposition import NMF
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

warnings.filterwarnings('ignore')

In [ ]:
# Загрузка датасета

df = pd.read_csv('movies_final_cleaned.csv')
required_columns = ['rank', 'title', 'year', 'country', 'genres', 'description', 'rating_kp', 'rating_imdb', 'votes_imdb']

missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"Отсутствуют обязательные колонки: {missing_columns}")

df = df[required_columns].copy()

print(f"Размер датасета: {df.shape}")
print("
Колонки:")
print(df.columns.tolist())

In [ ]:
# Базовая информация и статистика
print("Первые строки:")
display(df.head())

print("
Информация о датасете:")
df.info()

print("
Базовая статистика:")
display(df.describe(include='all').T)

## 2. Text Preprocessing & Vectorization

In [ ]:
# Подготовка русских стоп-слов
try:
    stopwords.words('russian')
except LookupError:
    nltk.download('stopwords')

russian_stopwords = set(stopwords.words('russian'))
morph = pymorphy3.MorphAnalyzer()

In [ ]:
# Функции предобработки текста

def fun_punctuation_text(text):
    text = '' if pd.isna(text) else str(text)
    text = text.lower()
    text = re.sub(r'[^а-яa-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def fun_lemmatizing_text(text):
    if not text:
        return ''
    return ' '.join(morph.parse(token)[0].normal_form for token in text.split())


def fun_tokenize(text):
    if not text:
        return ''
    tokens = [token for token in text.split() if token not in russian_stopwords and len(token) > 2]
    return ' '.join(tokens)


def preprocess_text(text):
    text = fun_punctuation_text(text)
    text = fun_lemmatizing_text(text)
    text = fun_tokenize(text)
    return text

In [ ]:
# Полная обработка описаний

df['description_processed'] = df['description'].fillna('').apply(preprocess_text)

print("Пример обработки:")
example_cols = ['description', 'description_processed']
display(df[example_cols].head(3))

In [ ]:
# TF-IDF векторизация

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 3)
)

X_tfidf = tfidf.fit_transform(df['description_processed'])

print(f"Форма TF-IDF матрицы: {X_tfidf.shape}")

## 3. Topic Modeling (NMF, n_components=7)

In [ ]:
n_topics = 7
nmf = NMF(n_components=n_topics, random_state=42, init='nndsvda', max_iter=300)
W = nmf.fit_transform(X_tfidf)
H = nmf.components_

feature_names = tfidf.get_feature_names_out()

def print_top_words(model, feature_names, n_top_words=12):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        print(f"Тема {topic_idx}: {', '.join(top_words)}")

print_top_words(nmf, feature_names, n_top_words=12)

In [ ]:
# Назначение темы каждому документу

df['topic'] = np.argmax(W, axis=1)
print("Распределение тем:")
print(df['topic'].value_counts().sort_index())

## 4. Clustering Optimization (K=2..10)

In [ ]:
ks = list(range(2, 11))
wcss = []
sil_scores = []

for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(W)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(W, labels))

# Эвристика локтя: максимальная дистанция до линии между крайними точками
x = np.array(ks)
y = np.array(wcss)
line_vec = np.array([x[-1] - x[0], y[-1] - y[0]])
line_len = np.linalg.norm(line_vec)
if line_len == 0:
    elbow_k = ks[0]
else:
    distances = []
    for xi, yi in zip(x, y):
        point_vec = np.array([xi - x[0], yi - y[0]])
        area = abs(np.cross(line_vec, point_vec))
        distances.append(area / line_len)
    elbow_k = ks[int(np.argmax(distances))]

optimal_k = ks[int(np.argmax(sil_scores))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ks, wcss, marker='o')
axes[0].set_title('Elbow Method (WCSS)')
axes[0].set_xlabel('K')
axes[0].set_ylabel('WCSS')
axes[0].scatter([elbow_k], [wcss[ks.index(elbow_k)]], color='red', zorder=3)
axes[0].annotate(f'Elbow: K={elbow_k}', (elbow_k, wcss[ks.index(elbow_k)]), textcoords='offset points', xytext=(10, -15))

axes[1].plot(ks, sil_scores, marker='o')
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette')
axes[1].scatter([optimal_k], [sil_scores[ks.index(optimal_k)]], color='green', zorder=3)
axes[1].annotate(f'Optimal: K={optimal_k}', (optimal_k, sil_scores[ks.index(optimal_k)]), textcoords='offset points', xytext=(10, -15))

plt.tight_layout()
plt.show()

print(f'Elbow K (WCSS): {elbow_k}')
print(f'Optimal K (Silhouette): {optimal_k}')

## 5. Classification Models

In [ ]:
# Делим на train/test 70/30
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf,
    df['topic'],
    test_size=0.30,
    random_state=42,
    stratify=df['topic']
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=400, random_state=42),
    'Random Forest Classifier': RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7, algorithm='brute', metric='cosine')
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    trained_models[name] = model

    print(f"
{'='*80}
{name}
{'='*80}")
    print(classification_report(y_test, preds, digits=4))

## 6. Model Persistence

In [ ]:
rf_model = trained_models['Random Forest Classifier']

joblib.dump(rf_model, 'model_rf.pkl')
joblib.dump(tfidf, 'vectorizer.pkl')

rf_preds = rf_model.predict(X_tfidf)
rf_proba = rf_model.predict_proba(X_tfidf)

df['predicted_topic'] = rf_preds
df['predicted_probability'] = rf_proba.max(axis=1)

df.to_csv('data_movies_classified.csv', index=False)

print('Сохранено: model_rf.pkl')
print('Сохранено: vectorizer.pkl')
print('Сохранено: data_movies_classified.csv')

## 7. Model Validation: predict_topic() on new texts

In [ ]:
def predict_topic(text):
    text_processed = preprocess_text(text)
    text_vector = tfidf.transform([text_processed])
    probabilities = rf_model.predict_proba(text_vector)[0]
    pred_class = int(np.argmax(probabilities))
    pred_proba = float(probabilities[pred_class])
    return pred_class, pred_proba

In [ ]:
sample_descriptions = [
    'Опытный детектив расследует серию загадочных убийств в мегаполисе.',
    'Молодые люди отправляются в опасное путешествие через океан и острова.',
    'Семья пытается сохранить отношения после переезда в другой город.',
    'Во время войны солдатам предстоит выполнить почти невозможную миссию.',
    'Ученый строит машину времени и меняет ход истории.'
]

results = []
for text in sample_descriptions:
    label, proba = predict_topic(text)
    results.append({
        'description': text,
        'predicted_topic': label,
        'probability': round(proba, 4)
    })

display(pd.DataFrame(results))